<a href="https://colab.research.google.com/github/quadrata8902-maker/Source-Code-of-Affine-Line-Universal-Cycle/blob/main/algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import itertools

# ==========================================
# Section 1: Finite Fields & Geometry
# ==========================================

class FiniteField:
    def __init__(self, q):
        self.q = q
    def add(self, a, b): raise NotImplementedError
    def sub(self, a, b): raise NotImplementedError
    def mul(self, a, b): raise NotImplementedError
    def inv(self, a):    raise NotImplementedError
    def vec_sub(self, v1, v2): return tuple(self.sub(x, y) for x, y in zip(v1, v2))
    def vec_mul_scalar(self, v, s): return tuple(self.mul(x, s) for x in v)

class PrimeField(FiniteField):
    def add(self, a, b): return (a + b) % self.q
    def sub(self, a, b): return (a - b) % self.q
    def mul(self, a, b): return (a * b) % self.q
    def inv(self, a):
        if a == 0: raise ValueError("Zero has no inverse.")
        return pow(a, self.q - 2, self.q)

class ExtensionField(FiniteField):
    """GF(p^k) arithmetic via polynomial representation."""
    def __init__(self, p, k, poly_coeffs):
        self.p, self.k, self.q = p, k, p ** k
        self.poly = poly_coeffs
        super().__init__(self.q)

        self._add_table = [[0] * self.q for _ in range(self.q)]
        self._mul_table = [[0] * self.q for _ in range(self.q)]
        self._inv_table = [0] * self.q
        self._build_tables()

    def _to_poly(self, n):
        return [(n // (self.p ** i)) % self.p for i in range(self.k)]

    def _from_poly(self, coeffs):
        return sum(c * (self.p ** i) for i, c in enumerate(coeffs))

    def _poly_add(self, n1, n2):
        c1, c2 = self._to_poly(n1), self._to_poly(n2)
        return self._from_poly([(x + y) % self.p for x, y in zip(c1, c2)])

    def _poly_mul(self, n1, n2):
        c1, c2 = self._to_poly(n1), self._to_poly(n2)
        deg_res = 2 * (self.k - 1)
        res_coeffs = [0] * (deg_res + 1)

        for i, j in itertools.product(range(self.k), repeat=2):
            res_coeffs[i + j] = (res_coeffs[i + j] + c1[i] * c2[j]) % self.p

        irr = list(reversed(self.poly))
        deg_irr = len(irr) - 1

        for i in range(deg_res, deg_irr - 1, -1):
            if res_coeffs[i] != 0:
                inv_lead = pow(irr[-1], self.p - 2, self.p)
                factor = (res_coeffs[i] * inv_lead) % self.p
                for j in range(len(irr)):
                    target_idx = i - deg_irr + j
                    res_coeffs[target_idx] = (res_coeffs[target_idx] - factor * irr[j]) % self.p

        return self._from_poly(res_coeffs[:self.k])

    def _build_tables(self):
        for i, j in itertools.product(range(self.q), repeat=2):
            self._add_table[i][j] = self._poly_add(i, j)
            self._mul_table[i][j] = self._poly_mul(i, j)

        # Isolate loops to ensure proper inverse mapping for all elements
        for i in range(1, self.q):
            for j in range(1, self.q):
                if self._mul_table[i][j] == 1:
                    self._inv_table[i] = j
                    break

    def add(self, a, b): return self._add_table[a][b]
    def sub(self, a, b):
        neg_b = self._from_poly([(self.p - x) % self.p for x in self._to_poly(b)])
        return self._add_table[a][neg_b]
    def mul(self, a, b): return self._mul_table[a][b]
    def inv(self, a):
        if a == 0: raise ValueError("Zero has no inverse.")
        return self._inv_table[a]
    def print_mapping_info(self):
        def format_poly(coeffs, var="\u03B1"):
            terms = []
            for i, c in enumerate(coeffs):
                if c == 0: continue
                if i == 0:
                    terms.append(str(c))
                elif i == 1:
                    terms.append(f"{c if c > 1 else ''}{var}")
                else:
                    terms.append(f"{c if c > 1 else ''}{var}^{i}")
            return " + ".join(reversed(terms)) if terms else "0"

        poly_str = format_poly(self.poly, "x")
        print(f"\n[Field Info] F_{self.q} ≅ F_{self.p}[x]/<{poly_str}>")

        print(f"We encode each element of F_{self.q} by an integer in [0, {self.q-1}] as follows:")
        print(f"Write a polynomial")
        print(f"    f(x) = c_0 + c_1 x + ... + c_{self.k-1} x^{self.k-1},")
        print(f"with coefficients c_i ∈ [0, {self.p-1}].")

        print(f"Define the encoding by evaluating at x = {self.p}:")
        print(f"    m = f({self.p}) = c_0 + c_1·{self.p} + ... + c_{self.k-1}·{self.p}^{self.k-1}.")

        print(f"This gives a bijection between F_{self.q} and integers 0 to {self.q-1}.")
        print("-" * 65)

def create_field(q):
    if q < 2: raise ValueError("q must be >= 2")

    factors = []
    d, temp = 2, q
    while d * d <= temp:
        while temp % d == 0:
            factors.append(d)
            temp //= d
        d += 1
    if temp > 1: factors.append(temp)

    unique_factors = sorted(list(set(factors)))
    if len(unique_factors) != 1: raise ValueError(f"q={q} is not a prime power.")

    p, k = unique_factors[0], len(factors)
    if k == 1: return PrimeField(q)

    poly_db = {
        4: [1, 1, 1], 8: [1, 0, 1, 1], 9: [1, 1, 2], 16: [1, 0, 0, 1, 1],
        25: [1, 0, 2], 27: [1, 0, 2, 1], 32: [1, 0, 0, 1, 0, 1],
        49: [1, 1, 0], 64: [1, 0, 0, 0, 0, 1, 1], 81: [1, 0, 0, 1, 2], 125: [1, 0, 1, 2],
    }
    if q in poly_db: return ExtensionField(p, k, poly_db[q])
    raise NotImplementedError(f"Irreducible polynomial for q={q} not defined.")

class AffinePoint:
    def __init__(self, coords): self.coords = tuple(coords)
    def __repr__(self): return str(self.coords)
    def __eq__(self, other): return isinstance(other, AffinePoint) and self.coords == other.coords
    def __hash__(self): return hash(self.coords)

class Direction:
    def __init__(self, vector, field):
        self.vector, self.field = tuple(vector), field
        self.normalize()

    def normalize(self):
        if all(x == 0 for x in self.vector): raise ValueError("Zero direction vector.")
        first_nonzero = next((x for x in self.vector if x != 0), None)
        if first_nonzero == 1:
            self.canonical_vector = self.vector
        else:
            inv = self.field.inv(first_nonzero)
            self.canonical_vector = tuple(self.field.mul(x, inv) for x in self.vector)

    def __repr__(self): return f"[{', '.join(map(str, self.canonical_vector))}]"
    def __eq__(self, other): return isinstance(other, Direction) and self.canonical_vector == other.canonical_vector
    def __hash__(self): return hash(self.canonical_vector)

class LineID:
    def __init__(self, dir_vec, pt_coords):
        self.dir_vec, self.pt_coords = tuple(dir_vec), tuple(pt_coords)
    def __repr__(self): return f"Dir{self.dir_vec}, Pt{self.pt_coords}"
    def __eq__(self, other): return isinstance(other, LineID) and self.dir_vec == other.dir_vec and self.pt_coords == other.pt_coords
    def __hash__(self): return hash((self.dir_vec, self.pt_coords))

# ==========================================
# Section 2: Validator
# ==========================================

def get_canonical_line_id(node1, node2, field):
    """Safely derive the unique Affine Line ID from two adjacent graph nodes."""
    if isinstance(node1, AffinePoint) and isinstance(node2, Direction):
        pt, dir_vec = node1.coords, node2.canonical_vector
    elif isinstance(node1, Direction) and isinstance(node2, AffinePoint):
        pt, dir_vec = node2.coords, node1.canonical_vector
    elif isinstance(node1, AffinePoint) and isinstance(node2, AffinePoint):
        pt = node1.coords
        raw_diff = field.vec_sub(node2.coords, node1.coords)
        if all(x == 0 for x in raw_diff): raise ValueError(f"Invalid Frame: Coinciding points {node1}")
        dir_vec = Direction(raw_diff, field).canonical_vector
    else:
        raise ValueError("Invalid Frame: Consecutive directions are not allowed.")

    # Shift point to standard representative using the pivot dimension
    pivot_idx = next((i for i, x in enumerate(dir_vec) if x != 0), -1)
    shift_vec = field.vec_mul_scalar(dir_vec, pt[pivot_idx])
    canonical_pt = field.vec_sub(pt, shift_vec)
    return LineID(dir_vec, canonical_pt)

def validate_u_cycle(sequence, n, q, field, verbose=False):
    expected_count = ((q**n - 1) // (q - 1)) * (q**(n-1))
    seen_lines = set()
    seq_len = len(sequence)
    if seq_len == 0: return False

    for i in range(seq_len):
        n1, n2 = sequence[i], sequence[(i + 1) % seq_len]
        try:
            line_id = get_canonical_line_id(n1, n2, field)
            if line_id in seen_lines:
                print(f"[Error] Duplicate line at step {i}: {line_id}")
                return False
            seen_lines.add(line_id)
        except ValueError as e:
            print(f"[Error] Step {i}: {e}")
            return False

    if len(seen_lines) == expected_count:
        print(f"Validation Passed: All {expected_count} lines perfectly covered.")
        return True

    print(f"Validation Failed: Expected {expected_count}, got {len(seen_lines)}.")
    return False

# ==========================================
# Section 3: Math Helpers
# ==========================================

def is_linearly_independent(basis, new_vec, field):
    """Gaussian elimination for linear independence check."""
    matrix = [list(b) for b in basis] + [list(new_vec)]
    rows, cols = len(matrix), len(new_vec)
    pivot_row = 0

    for col in range(cols):
        if pivot_row >= rows: break
        pivot = next((r for r in range(pivot_row, rows) if matrix[r][col] != 0), -1)
        if pivot == -1: continue

        matrix[pivot_row], matrix[pivot] = matrix[pivot], matrix[pivot_row]
        inv = field.inv(matrix[pivot_row][col])
        for r in range(rows):
            if r != pivot_row and matrix[r][col] != 0:
                factor = field.mul(matrix[r][col], inv)
                for c in range(col, cols):
                    matrix[r][c] = field.sub(matrix[r][c], field.mul(factor, matrix[pivot_row][c]))
        pivot_row += 1
    return any(x != 0 for x in matrix[-1])

def get_complement_basis(u_a, u_b, n, q, field):
    basis, w0_basis = [u_a, u_b], []
    for i in range(n):
        vec = tuple(1 if j == i else 0 for j in range(n))
        if is_linearly_independent(basis, vec, field):
            basis.append(vec)
            w0_basis.append(vec)
        if len(basis) == n: break
    return w0_basis

def generate_rep_set_points(u_a, u_b, w0_basis, field):
    v_sum = tuple(field.add(x, y) for x, y in zip(u_a, u_b))
    points = []
    for c in range(field.q):
        vec_c = field.vec_mul_scalar(v_sum, c)
        for coeffs in itertools.product(range(field.q), repeat=len(w0_basis)):
            vec_w = tuple([0] * len(u_a))
            for i, coeff in enumerate(coeffs):
                term = field.vec_mul_scalar(w0_basis[i], coeff)
                vec_w = tuple(field.add(x, y) for x, y in zip(vec_w, term))
            points.append(AffinePoint(tuple(field.add(x, y) for x, y in zip(vec_c, vec_w))))
    return points

def get_sorted_directions(n, q, field):
    raw_vectors = [v for v in itertools.product(range(q), repeat=n) if any(x != 0 for x in v)]
    unique_dirs = {Direction(vec, field) for vec in raw_vectors}
    return sorted(list(unique_dirs), key=lambda d: d.canonical_vector)

# ==========================================
# Section 4: Core Constructions
# ==========================================

def construct_pairwise_cycle(w_a, w_b, n, q, field):
    u_a, u_b = w_a.canonical_vector, w_b.canonical_vector
    w0_basis = get_complement_basis(u_a, u_b, n, q, field)
    A_rep = generate_rep_set_points(u_a, u_b, w0_basis, field)
    sequence = []

    # Even Parity: straightforward alternation
    if len(A_rep) % 2 == 0:
        for i, point in enumerate(A_rep):
            sequence.extend([point, w_a if i % 2 == 0 else w_b])
        return sequence

    # Odd Parity: origin wrap-around required
    zero_point = AffinePoint(tuple([0]*n))
    start_point = AffinePoint(tuple(field.add(x, y) for x, y in zip(u_a, u_b)))

    A_rep_list = A_rep.copy()
    if zero_point in A_rep_list: A_rep_list.remove(zero_point)
    if start_point in A_rep_list: A_rep_list.remove(start_point)

    # Reconstruct paper's exact logic: 0 -> a*u1 -> w* -> [l1] -> ...
    sequence = [zero_point, AffinePoint(u_a), start_point, w_a]
    for i, point in enumerate(A_rep_list):
        sequence.extend([point, w_b if i % 2 == 0 else w_a])

    return sequence

def construct_triple_cycle_2d_base(q, field):
    wa, wb, wc = Direction((0, 1), field), Direction((1, 0), field), Direction((1, 1), field)
    sequence = []

    if q % 2 == 0:
        for u in range(0, q, 2):
            u_plus_1 = field.add(u, 1)
            sequence.extend([
                AffinePoint((u, u_plus_1)), wa,
                AffinePoint((u_plus_1, 0)), wc,
                AffinePoint((0, u)), wb
            ])
    else:
        # Base Eulerian kernel mapped with strict field arithmetics
        q_minus_1 = field.sub(0, 1)
        two = field.add(1, 1)

        sequence.extend([
            AffinePoint((0, 1)), AffinePoint((0, q_minus_1)), wc,
            AffinePoint((two, 0)), AffinePoint((0, 0)), AffinePoint((1, 1)), wa,
            AffinePoint((two, two)), wb
        ])

        elements = list(range(3, q))
        for i in range(0, len(elements), 2):
            u, v = elements[i], elements[i+1]
            sequence.extend([
                AffinePoint((u, u)), wa,
                AffinePoint((v, 0)), wc,
                AffinePoint((field.add(u, v), v)), wb
            ])
    return sequence

def lift_cycle(base_sequence, n_target, q, field):
    # Lift elements recursively while preserving pivot connections (Figure-8)
    pivot_dir = next((node for node in base_sequence if isinstance(node, Direction)), None)
    if not pivot_dir: raise ValueError("Lifting failed: No direction found.")

    final_sequence = []
    for k in range(q):
        layer_seq = []
        for node in base_sequence:
            if isinstance(node, AffinePoint):
                layer_seq.append(AffinePoint(node.coords + (k,)))
            elif isinstance(node, Direction):
                layer_seq.append(Direction(node.vector + (0,), field))

        lifted_pivot = Direction(pivot_dir.vector + (0,), field)
        pivot_idx = layer_seq.index(lifted_pivot)
        final_sequence.extend(layer_seq[pivot_idx:] + layer_seq[:pivot_idx])

    return final_sequence

# ==========================================
# Section 5: Eulerian Circuit Merger
# ==========================================

def generate_u_cycle(n, q):
    field = create_field(q)

    if isinstance(field, ExtensionField):
        field.print_mapping_info()

    all_dirs = get_sorted_directions(n, q, field)
    remaining_dirs = all_dirs.copy()

    final_sequence = []
    zero_point = AffinePoint(tuple([0]*n))

    # Phase 1: Triple Construction
    if len(all_dirs) % 2 != 0:
        target_triple = [Direction(v + tuple([0]*(n-2)), field) for v in [(0,1), (1,0), (1,1)]]
        for target in target_triple:
            if target in remaining_dirs: remaining_dirs.remove(target)

        triple_seq = construct_triple_cycle_2d_base(q, field)
        for curr_n in range(3, n + 1):
            triple_seq = lift_cycle(triple_seq, curr_n, q, field)

        idx_0 = triple_seq.index(zero_point)
        rotated_triple = triple_seq[idx_0:] + triple_seq[:idx_0]
        final_sequence.extend(rotated_triple)

    # Phase 2: Pairwise Construction
    pairs = [(remaining_dirs[i], remaining_dirs[i+1]) for i in range(0, len(remaining_dirs), 2)]

    for w_a, w_b in pairs:
        cycle = construct_pairwise_cycle(w_a, w_b, n, q, field)
        idx_0 = cycle.index(zero_point)
        rotated_cycle = cycle[idx_0:] + cycle[:idx_0]
        final_sequence.extend(rotated_cycle)

    return final_sequence

# ==========================================
# Section 6: Interactive Execution
# ==========================================

def print_sequence_preview(sequence, limit=20):
    length = len(sequence)
    print(f"\n[Sequence Preview] Total length: {length}")
    if length <= limit:
        print(sequence)
    else:
        half = limit // 2
        print(f"{sequence[:half]} ... ... {sequence[-half:]}")
        print(f"({length - limit} elements omitted)")

def interactive_mode():
    print("\n" + "="*50)
    print("  Affine Line U-Cycle Generator (Interactive CLI)")
    print("="*50)

    while True:
        try:
            n_input = input("\nEnter dimension n (e.g., 2, 3): ").strip()
            if n_input.lower() in ['exit', 'quit']: break
            if not n_input: continue

            q_input = input("Enter field size q (prime or prime power): ").strip()
            if q_input.lower() in ['exit', 'quit']: break

            n, q = int(n_input), int(q_input)
            if n < 2 or q < 2:
                print("Error: Both n and q must be >= 2.")
                continue

            try:
                final_seq = generate_u_cycle(n, q)
                print_sequence_preview(final_seq)

                total_lines = ((q**n - 1) // (q - 1)) * (q**(n-1))
                run_validation = True

                if total_lines > 1000:
                    check = input(f"Warning: Theoretical line count is {total_lines}. Run validation? (y/n): ")
                    if check.lower() != 'y': run_validation = False

                if run_validation:
                    field = create_field(q)
                    validate_u_cycle(final_seq, n, q, field, verbose=(total_lines < 100))

            except Exception as e:
                print(f"\n[Generation Error]: {e}")
                import traceback; traceback.print_exc()

        except ValueError:
            print("Invalid input format. Please enter integers.")
        except KeyboardInterrupt:
            print("\nProgram terminated.")
            break

if __name__ == "__main__":
    interactive_mode()


  Affine Line U-Cycle Generator (Interactive CLI)

Program terminated.
